# LSTM - Hybrid GA + Local Random Search

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import numpy as np
import json
import copy
import time
import random
from collections import Counter
from data_loader import load_clinc
from ga_optimizer import GeneticOptimizer

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

USE_SAMPLE = False
data = load_clinc(use_sample=USE_SAMPLE)
train_texts, train_labels = data['train_texts'], data['train_labels']
val_texts, val_labels = data['val_texts'], data['val_labels']
test_texts, test_labels = data['test_texts'], data['test_labels']
num_classes = data['num_classes']

cuda


15250 / 3100 / 5500, 151 classes (full)


In [2]:
class Vocabulary:
    def __init__(self, min_freq=2):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.min_freq = min_freq

    def build_vocab(self, texts):
        counts = Counter()
        for t in texts:
            counts.update(t.lower().split())
        idx = 2
        for word, cnt in counts.items():
            if cnt >= self.min_freq:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1

    def encode(self, text):
        return [self.word2idx.get(w, 1) for w in text.lower().split()]

    def __len__(self):
        return len(self.word2idx)


vocab = Vocabulary(min_freq=2)
vocab.build_vocab(train_texts)
vocab_size = len(vocab)
print(f"Vocab: {vocab_size}")

Vocab: 3137


In [3]:
class TextSequenceDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.texts[idx])), torch.LongTensor([self.labels[idx]])


def collate_batch(batch):
    seqs, labels = zip(*batch)
    seqs = [s if len(s) > 0 else torch.LongTensor([1]) for s in seqs]
    lengths = torch.LongTensor([len(s) for s in seqs])
    return pad_sequence(seqs, batch_first=True, padding_value=0), lengths, torch.cat(labels)


class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes,
                 num_layers=2, dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0,
                           bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), num_classes)

    def forward(self, sequences, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.embedding(sequences), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        if self.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]
        return self.fc(self.dropout(hidden))

In [4]:
PATIENCE = 5
MAX_EPOCHS = 30

train_ds = TextSequenceDataset(train_texts, train_labels, vocab)
val_ds = TextSequenceDataset(val_texts, val_labels, vocab)


def evaluate_hyperparams(genes):
    torch.manual_seed(42)
    np.random.seed(42)

    tr_loader = DataLoader(train_ds, batch_size=genes['batch_size'], shuffle=True, collate_fn=collate_batch)
    vl_loader = DataLoader(val_ds, batch_size=genes['batch_size'], shuffle=False, collate_fn=collate_batch)

    model = LSTMClassifier(
        vocab_size, genes['embedding_dim'], genes['hidden_dim'], num_classes,
        num_layers=genes['num_layers'], dropout=genes['dropout'],
        bidirectional=True
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=genes['learning_rate'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

    best_f1, patience_counter = 0, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        for seqs, lengths, labels in tr_loader:
            seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(seqs, lengths), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        model.eval()
        all_preds = []
        with torch.no_grad():
            for seqs, lengths, labels in vl_loader:
                seqs, lengths = seqs.to(device), lengths.to(device)
                all_preds.extend(model(seqs, lengths).argmax(1).cpu().numpy())

        vl_f1 = f1_score(val_labels, all_preds, average='macro', zero_division=0)
        scheduler.step(vl_f1)

        if vl_f1 > best_f1:
            best_f1, patience_counter = vl_f1, 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            break

    del model, optimizer, scheduler, criterion
    torch.cuda.empty_cache()

    return best_f1, {"val_f1": round(best_f1, 4), "epochs": epoch + 1}

## Phase 1: Genetic Algorithm

In [5]:
search_space = {
    'embedding_dim': [64, 128, 256],
    'hidden_dim': [128, 256, 512],
    'num_layers': [1, 2, 3],
    'dropout': [0.1, 0.2, 0.3, 0.4, 0.5],
    'learning_rate': [0.0001, 0.0005, 0.001, 0.005],
    'batch_size': [32, 64, 128]
}

TOTAL_BUDGET = 72

ga = GeneticOptimizer(
    search_space=search_space,
    fitness_fn=evaluate_hyperparams,
    population_size=7,
    n_generations=8,
    crossover_rate=0.8,
    mutation_rate=0.2,
    tournament_size=3,
    elitism_count=2,
    seed=42
)

ga_results = ga.run()
ga_best = ga_results['best_genes']
ga_evals = ga_results['total_evaluations']
print(f"\nPhase 1 (GA): Best F1={ga_results['best_fitness']:.4f}, Evals={ga_evals}")

local_budget = TOTAL_BUDGET - ga_evals

def build_neighborhood(search_space, best_genes, radius=1):
    neighborhood = {}
    for param, values in search_space.items():
        idx = values.index(best_genes[param])
        lo = max(0, idx - radius)
        hi = min(len(values) - 1, idx + radius)
        neighborhood[param] = values[lo:hi + 1]
    return neighborhood

neighborhood = build_neighborhood(search_space, ga_best)
for k, v in neighborhood.items():
    print(f"  {k}: {v}")

random.seed(42)
local_results = []
best_local_f1, best_local_params = ga_results['best_fitness'], ga_best
local_start = time.time()

for i in range(local_budget):
    params = {k: random.choice(v) for k, v in neighborhood.items()}
    eval_start = time.time()
    f1, metrics = evaluate_hyperparams(params)
    eval_time = time.time() - eval_start
    local_results.append({"params": params, "val_f1": round(f1, 4),
                          "eval_time": round(eval_time, 2)})
    if f1 > best_local_f1:
        best_local_f1, best_local_params = f1, params
    print(f"  Local {i+1}/{local_budget}  f1={f1:.4f}")

local_time = time.time() - local_start

hybrid_best_f1 = best_local_f1
hybrid_best_params = best_local_params
total_evals = ga_evals + local_budget

print(f"\nGA best val F1:     {ga_results['best_fitness']:.4f}")
print(f"Local best val F1:  {best_local_f1:.4f}")
print(f"Hybrid best val F1: {hybrid_best_f1:.4f}")
print(f"Hybrid best params: {hybrid_best_params}")
print(f"Total evals: {total_evals}")
print(f"Local phase time: {local_time:.1f}s")

Gen  1/8  best=0.8925  avg=0.8624  global_best=0.8925
Gen  2/8  best=0.8925  avg=0.8765  global_best=0.8925
Gen  3/8  best=0.8952  avg=0.8781  global_best=0.8952
Gen  4/8  best=0.8952  avg=0.8908  global_best=0.8952
Gen  5/8  best=0.8952  avg=0.8866  global_best=0.8952
Gen  6/8  best=0.8978  avg=0.8941  global_best=0.8978
Gen  7/8  best=0.8978  avg=0.8931  global_best=0.8978
Gen  8/8  best=0.8978  avg=0.8946  global_best=0.8978

Phase 1 (GA): Best F1=0.8984, Evals=43
  embedding_dim: [128, 256]
  hidden_dim: [128, 256, 512]
  num_layers: [1, 2]
  dropout: [0.4, 0.5]
  learning_rate: [0.0005, 0.001, 0.005]
  batch_size: [32, 64]
  Local 1/29  f1=0.8803
  Local 2/29  f1=0.8675
  Local 3/29  f1=0.8804
  Local 4/29  f1=0.8832
  Local 5/29  f1=0.8876
  Local 6/29  f1=0.8776
  Local 7/29  f1=0.8873
  Local 8/29  f1=0.8495
  Local 9/29  f1=0.8675
  Local 10/29  f1=0.8804
  Local 11/29  f1=0.8765
  Local 12/29  f1=0.8636
  Local 13/29  f1=0.8891
  Local 14/29  f1=0.8806
  Local 15/29  f1=0.889

## Test Evaluation

In [6]:
best = hybrid_best_params
torch.manual_seed(42)
np.random.seed(42)

tr_loader = DataLoader(train_ds, batch_size=best['batch_size'], shuffle=True, collate_fn=collate_batch)
vl_loader = DataLoader(val_ds, batch_size=best['batch_size'], shuffle=False, collate_fn=collate_batch)
te_loader = DataLoader(TextSequenceDataset(test_texts, test_labels, vocab),
                       batch_size=best['batch_size'], shuffle=False, collate_fn=collate_batch)

model = LSTMClassifier(
    vocab_size, best['embedding_dim'], best['hidden_dim'], num_classes,
    num_layers=best['num_layers'], dropout=best['dropout'], bidirectional=True
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=best['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

best_f1, best_state, patience_counter = 0, None, 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for seqs, lengths, labels in tr_loader:
        seqs, lengths, labels = seqs.to(device), lengths.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(seqs, lengths), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

    model.eval()
    vl_preds = []
    with torch.no_grad():
        for seqs, lengths, labels in vl_loader:
            vl_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

    vl_f1 = f1_score(val_labels, vl_preds, average='macro', zero_division=0)
    scheduler.step(vl_f1)

    if vl_f1 > best_f1:
        best_f1, best_state, patience_counter = vl_f1, copy.deepcopy(model.state_dict()), 0
    else:
        patience_counter += 1
    if patience_counter >= PATIENCE:
        break

model.load_state_dict(best_state)
model.eval()
te_preds = []
with torch.no_grad():
    for seqs, lengths, labels in te_loader:
        te_preds.extend(model(seqs.to(device), lengths.to(device)).argmax(1).cpu().numpy())

acc = accuracy_score(test_labels, te_preds)
p = precision_score(test_labels, te_preds, average='macro', zero_division=0)
r = recall_score(test_labels, te_preds, average='macro', zero_division=0)
f1_m = f1_score(test_labels, te_preds, average='macro', zero_division=0)
f1_w = f1_score(test_labels, te_preds, average='weighted', zero_division=0)

print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1 macro:   {f1_m:.4f}")
print(f"F1 weighted:{f1_w:.4f}")

del model, optimizer, scheduler, criterion, best_state
torch.cuda.empty_cache()

Accuracy:   0.8464
Precision:  0.8496
Recall:     0.9066
F1 macro:   0.8727
F1 weighted:0.8425


## Save Results

In [8]:
ga_time = ga_results['total_time_seconds']
total_time = ga_time + local_time

results = {
    "model_type": "LSTM",
    "optimization": "hybrid_ga_local_random",
    "best_hyperparameters": hybrid_best_params,
    "total_evaluations": total_evals,
    "search_time_seconds": round(total_time, 2),
    "ga_phase": {
        "evaluations": ga_evals,
        "time_seconds": round(ga_time, 2),
        "best_val_f1": round(ga_results['best_fitness'], 4)
    },
    "local_phase": {
        "evaluations": local_budget,
        "time_seconds": round(local_time, 2),
        "best_val_f1": round(best_local_f1, 4)
    },
    "test_metrics": {
        "accuracy": round(acc, 4),
        "macro_precision": round(p, 4),
        "macro_recall": round(r, 4),
        "macro_f1": round(f1_m, 4),
        "weighted_f1": round(f1_w, 4)
    },
    "predictions": [int(x) for x in te_preds],
    "ga_history": ga_results['history'],
    "local_results": local_results,
    "all_ga_evaluations": ga.all_evaluations
}

with open('results/results_lstm_hybrid.json', 'w') as f:
    json.dump(results, f, indent=4, default=str)

print("Saved: results/results_lstm_hybrid.json")

Saved: results/results_lstm_hybrid.json
